# Topic 9 — Classification Evaluation
### Theory → tiny example → experiment.

**Why this topic before more classifiers?** On imbalanced data (like cyberbullying detection, where
most posts are NOT bullying), a model can score 90%+ **accuracy** just by always predicting "not
bullying" — while being completely useless. You need better metrics to catch this.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
    classification_report
)

rng = np.random.default_rng(0)

## 1. The confusion matrix

For binary classification (positive = "bullying", negative = "not bullying"):

|                | Predicted Negative | Predicted Positive |
|----------------|---------------------|---------------------|
| **Actual Negative** | True Negative (TN)  | False Positive (FP) |
| **Actual Positive** | False Negative (FN) | True Positive (TP)  |

- **TP**: correctly caught bullying
- **TN**: correctly identified non-bullying
- **FP**: flagged non-bullying as bullying (false alarm)
- **FN**: missed actual bullying (the dangerous kind of mistake in this domain)

In [ ]:
# Simulate an IMBALANCED dataset like a real cyberbullying dataset: 90% not-bullying, 10% bullying
X, y = make_classification(
    n_samples=1000, n_features=5, weights=[0.9, 0.1],
    n_informative=3, n_redundant=1, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

print("class balance in test set:", np.bincount(y_test) / len(y_test))

clf = LogisticRegression().fit(X_train, y_train)
y_pred = clf.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print("confusion matrix:\n", cm)

ConfusionMatrixDisplay(cm, display_labels=["not_bullying", "bullying"]).plot(cmap="Blues")
plt.title("Confusion matrix")
plt.show()

## 2. Accuracy — and why it can lie to you

`accuracy = (TP + TN) / total`. Simple, but misleading on imbalanced data.

In [ ]:
acc = accuracy_score(y_test, y_pred)
print("model accuracy:", acc)

# Compare to a "dumb" baseline that always predicts the majority class
dumb_pred = np.zeros_like(y_test)   # always predicts 0 (not_bullying)
dumb_acc = accuracy_score(y_test, dumb_pred)
print("'always predict not-bullying' accuracy:", dumb_acc)
# If these two numbers are close, accuracy alone isn't telling you much about real skill.

## 3. Precision, Recall, F1, Specificity

- **Precision** = TP / (TP + FP) — "of everything I flagged as bullying, how much really was?"
  High precision = few false alarms.
- **Recall** (a.k.a. sensitivity) = TP / (TP + FN) — "of all real bullying, how much did I catch?"
  High recall = few missed cases.
- **F1** = harmonic mean of precision and recall — a single number balancing both.
- **Specificity** = TN / (TN + FP) — "of all real non-bullying, how much did I correctly leave alone?"

**Precision vs recall tradeoff for cyberbullying detection**: missing real bullying (low recall) is
often worse than a few false alarms (low precision) — but that's a judgment call for your specific
application, not something accuracy alone can tell you.

In [ ]:
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)

print(f"precision:   {precision:.3f}")
print(f"recall:      {recall:.3f}")
print(f"f1:          {f1:.3f}")
print(f"specificity: {specificity:.3f}")

print()
print(classification_report(y_test, y_pred, target_names=["not_bullying", "bullying"]))

## 4. ROC curve & ROC-AUC

The **ROC curve** plots True Positive Rate (recall) vs False Positive Rate, at every possible
decision threshold. **ROC-AUC** (area under that curve) summarizes overall ranking ability in one
number: 1.0 = perfect, 0.5 = no better than random guessing.

In [ ]:
y_proba = clf.predict_proba(X_test)[:, 1]   # probability of class 1

fpr, tpr, roc_thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="random guessing")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC curve")
plt.legend()
plt.show()

## 5. Precision-Recall curve & PR-AUC

**On imbalanced data, the PR curve is usually more informative than ROC.** ROC-AUC can look
deceptively good even for a weak minority-class detector, because the negative class dominates the
False Positive Rate calculation. PR curves focus entirely on how well you handle the positive
(minority) class.

In [ ]:
precisions, recalls, pr_thresholds = precision_recall_curve(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)

plt.figure(figsize=(5, 5))
plt.plot(recalls, precisions, label=f"PR curve (AUC = {pr_auc:.3f})")
baseline = y_test.mean()   # a random classifier's PR curve is flat at this level
plt.axhline(baseline, linestyle="--", color="gray", label=f"random baseline ({baseline:.2f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall curve")
plt.legend()
plt.show()
# For imbalanced datasets like yours: prefer PR-AUC over ROC-AUC as your headline metric.

## 6. Threshold tuning

The default 0.5 threshold isn't always right. If missing bullying (false negatives) is more costly
than false alarms, LOWER the threshold to catch more positives (trading away some precision).

In [ ]:
for threshold in [0.3, 0.5, 0.7]:
    y_pred_t = (y_proba >= threshold).astype(int)
    p = precision_score(y_test, y_pred_t)
    r = recall_score(y_test, y_pred_t)
    f = f1_score(y_test, y_pred_t)
    print(f"threshold={threshold}: precision={p:.3f}, recall={r:.3f}, f1={f:.3f}")
# Lowering the threshold generally raises recall but lowers precision -- there's no free lunch,
# you choose the tradeoff based on what matters most for your use case.

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Retrain `clf` using class_weight="balanced" in LogisticRegression and compare precision/recall/F1
#    to the unweighted version above.
# 2. Sweep threshold from 0.1 to 0.9 in steps of 0.1 and plot precision & recall vs threshold on one chart.
# 3. Compute ROC-AUC and PR-AUC for the dumb "always predict 0" baseline's probabilities
#    (hint: use np.zeros(len(y_test)) as fake probabilities) and see how misleadingly high ROC-AUC
#    can look compared to PR-AUC on very imbalanced data.
# 4. Write one sentence: for YOUR cyberbullying paper, would you prioritize precision or recall? Why?

---
### Next up: **Topic 10 — K-Nearest Neighbors**.

Say "next" when you're ready.